# Preprocessing Teks Ulasan E-Commerce Berbahasa Indonesia

**Nama:** (Haikal Azzuhry)  
**NIM:** (24146058)

Dataset: [dipawidia/ecommerce-product-reviews-sentiment](https://huggingface.co/datasets/dipawidia/ecommerce-product-reviews-sentiment)

## 1. Install library

In [ ]:
!pip install datasets Sastrawi nltk pandas matplotlib

## 2. Load dataset dan informasi dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("dipawidia/ecommerce-product-reviews-sentiment")
df = ds["train"].to_pandas()

print("Jumlah data :", len(df))
print("Kolom       :", df.columns.tolist())
print()
df.info()
print()
df.head()

### Distribusi label sentimen (1 = positif, 0 = negatif)

In [ ]:
label_map = {0: "Negatif", 1: "Positif"}
print(df["sentimen"].value_counts())
print()
print((df["sentimen"].map(label_map).value_counts(normalize=True) * 100).round(2))

df["sentimen"].map(label_map).value_counts().plot(
    kind="bar", title="Distribusi Label Sentimen", rot=0)

## 3. Preprocessing teks

Tahapan: case folding, cleaning, tokenisasi, stopword removal, stemming. Kolom yang dipakai adalah `review` (bahasa Indonesia).

In [ ]:
import re, string, nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from functools import lru_cache

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

stop_words = set(stopwords.words("indonesian"))
# Kata negasi dipertahankan karena penting untuk makna sentimen
stop_words -= {"tidak", "bukan", "belum", "tak"}

stemmer = StemmerFactory().create_stemmer()

def case_folding(text):
    return str(text).lower()

def cleaning(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)                      # hapus URL
    text = re.sub(r"\d+", " ", text)                                   # hapus angka
    text = text.translate(str.maketrans("", "", string.punctuation))   # hapus tanda baca
    text = re.sub(r"[^a-z\s]", " ", text)                              # hapus karakter khusus/emoji
    return re.sub(r"\s+", " ", text).strip()

def tokenizing(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words]

@lru_cache(maxsize=None)
def stem_word(w):
    return stemmer.stem(w)

def stemming(tokens):
    return [stem_word(t) for t in tokens]

### Jalankan pipeline

Stemming bisa memakan waktu beberapa menit untuk 11 ribu baris. Mohon ditunggu.

In [ ]:
df["case_folding"] = df["review"].apply(case_folding)
df["cleaning"]     = df["case_folding"].apply(cleaning)
df["tokens"]       = df["cleaning"].apply(tokenizing)
df["stopword"]     = df["tokens"].apply(remove_stopwords)
df["stemming"]     = df["stopword"].apply(stemming)
df["hasil_akhir"]  = df["stemming"].apply(lambda x: " ".join(x))
print("Preprocessing selesai.")

## 4. 10 contoh teks sebelum dan sesudah preprocessing

In [ ]:
pd.set_option("display.max_colwidth", 150)
df[["review", "hasil_akhir"]].head(10)

### Contoh per tahap (3 data pertama)

In [ ]:
df[["review", "case_folding", "cleaning", "stopword", "stemming"]].head(3)

## 5. Pengaruh preprocessing terhadap kualitas data teks

*(Tulis penjelasan dengan kata-katamu sendiri di sini. Bahas pengaruh tiap tahap: case folding, cleaning, tokenisasi, stopword removal, dan stemming terhadap kualitas data teks.)*